# 🧠 Sam3
# source ~/venv/vlm/bin/activate

In [7]:
# Install once: !pip install open3d
import plotly.io as pio
pio.renderers.default = "notebook"  # or "notebook_connected" / "colab" depending on environment
import open3d as o3d
import numpy as np

pcd = o3d.io.read_point_cloud("dense_cloud.pcd")
print(f"Points: {len(pcd.points)}")
print(f"Has colors: {pcd.has_colors()}")
print(f"Bounding box: {pcd.get_axis_aligned_bounding_box()}")
pcd.colors = o3d.utility.Vector3dVector([])  # Remove colors

subset = pcd.random_down_sample(0.5)  # Optional: downsample for faster rendering


# Get accurate center
center = pcd.get_center()
print(f"Point cloud center: {center}") # [-0.01213967 -0.00221211  0.98910967]
print(f"Point count after subset: {len(subset.points)}")

# Visualize in notebook
o3d.visualization.draw_plotly([subset],
                              lookat=center,                # Critical: look at cloud center, not origin
                              front=[0.0, 0.0, -1.0],       # Camera looks along -Z (from +Z toward cloud)
                              up=[0.0, -1.0, 0.0],
)


ModuleNotFoundError: No module named 'plotly'

In [15]:
points = np.asarray(pcd.points)
print(f"Points shape: {points.shape}")
print(f"Point min/max: {points.min(axis=0)} / {points.max(axis=0)}")
print(f"Any NaN in points? {np.isnan(points).any()}")
print(f"Any Inf in points? {np.isinf(points).any()}")
print(f"Finite points count: {np.isfinite(points).all(axis=1).sum()} / {len(points)}")

Points shape: (833152, 3)
Point min/max: [-1.008124 -0.582293  0.707622] / [0.862096 0.592667 1.281139]
Any NaN in points? False
Any Inf in points? False
Finite points count: 833152 / 833152


In [1]:
import cv2
import numpy as np
import torch
from moge.model.v2 import MoGeModel
device = torch.device("cuda")

# pip install git+https://github.com/EasternJournalist/utils3d.git@3fab839f0be9931dac7c8488eb0e1600c236e183

In [2]:
model = MoGeModel.from_pretrained("yjh001/metricanything_student_pointmap").to(device)                             

# Read the input image and convert to tensor (3, H, W) with RGB values normalized to [0, 1]
#input_image = cv2.cvtColor(cv2.imread('/home/markpp/Desktop/droplet_segmentation/data/iphone/iphone_wet/frame_000061.png'), cv2.COLOR_BGR2RGB)          
input_image = cv2.cvtColor(cv2.imread('/home/markpp/DevOps/edge_ml/offline_demo/test_input/wet_crop_0.png'), cv2.COLOR_BGR2RGB)                       

input_tensor = torch.tensor(input_image / 255, dtype=torch.float32, device=device).permute(2, 0, 1)    

# Infer 
output = model.infer(input_tensor)
"""
`output` has keys "points", "depth", "mask" and "intrinsics",
The maps are in the same size as the input image. 
{
    "points": (H, W, 3),    # point map with metric scale in OpenCV camera coordinate system (x right, y down, z forward).
    "depth": (H, W),        # depth map
    "mask": (H, W),         # a binary mask for valid pixels. 
    "intrinsics": (3, 3),   # normalized camera intrinsics
}
"""
output["intrinsics"]

tensor([[0.7386, 0.0000, 0.5000],
        [0.0000, 0.8774, 0.5000],
        [0.0000, 0.0000, 1.0000]], device='cuda:0')

In [3]:
cv2.imshow("Mask", output["mask"].cpu().numpy().astype(np.uint8) * 255)
cv2.waitKey(0)
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/markpp/venv/meta/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/markpp/venv/meta/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/markpp/venv/meta/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/markpp/venv/meta/lib/python3.13/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/markpp/venv/meta/lib/pyt

In [4]:
def save_structured_pcd(points, filename, colors=None, mask=None):
    """
    Save a structured (dense) point cloud in PCD format with WIDTH/HEIGHT headers.
    
    Args:
        points: (H, W, 3) array of 3D coordinates
        filename: output .pcd filename
        colors: (H, W, 3) uint8 RGB image (optional)
        mask: (H, W) boolean array (optional); masked pixels are set to NaN
    """
    H, W, _ = points.shape
    
    # Apply mask: set invalid points to NaN to preserve grid structure
    if mask is not None:
        points = np.where(mask[..., None], points, np.nan)
        if colors is not None:
            colors = np.where(mask[..., None], colors, 0)
    
    # Reshape to linear arrays while preserving raster order (row-major)
    points_flat = points.reshape(-1, 3)
    
    # Build header
    header = [
        "# .PCD v0.7 - Point Cloud Data file format",
        "VERSION 0.7",
        "FIELDS x y z",
        "SIZE 4 4 4",
        "TYPE F F F",
        "COUNT 1 1 1",
        f"WIDTH {W}",
        f"HEIGHT {H}",
        "VIEWPOINT 0 0 0 1 0 0 0",
        f"POINTS {H * W}",
        "DATA ascii"
    ]
    
    # Add RGB field if colors provided
    if colors is not None:
        colors_flat = colors.reshape(-1, 3).astype(np.uint8)
        header[2] = "FIELDS x y z rgb"
        header[3] = "SIZE 4 4 4 4"
        header[4] = "TYPE F F F U"
        header[5] = "COUNT 1 1 1 1"
    
    # Write file
    with open(filename, 'w') as f:
        f.write('\n'.join(header) + '\n')
        
        if colors is not None:
            for (x, y, z), (r, g, b) in zip(points_flat, colors_flat):
                if np.isnan(x):
                    f.write("nan nan nan 0\n")
                else:
                    # Pack RGB into float (PCL convention)
                    rgb = (r << 16) | (g << 8) | b
                    f.write(f"{x:.6f} {y:.6f} {z:.6f} {rgb}\n")
        else:
            for x, y, z in points_flat:
                if np.isnan(x):
                    f.write("nan nan nan\n")
                else:
                    f.write(f"{x:.6f} {y:.6f} {z:.6f}\n")
    
    print(f"✓ Saved structured point cloud ({H}×{W}) to {filename}")
    print("  → Open in CloudCompare: maintains grid indexing via WIDTH/HEIGHT")

points = output["points"].cpu().numpy()      # (736, 1132, 3)
mask = output["mask"].cpu().numpy()          # (736, 1132), bool or 0/1
img_uint8 = (input_image * 255).astype(np.uint8) if input_image.max() <= 1.0 else input_image.astype(np.uint8)

save_structured_pcd(
    points=points,
    filename="dense_cloud.pcd",
    colors=img_uint8,
    mask=mask  # Invalid pixels become NaN but retain grid position
)

✓ Saved structured point cloud (2128×2528) to dense_cloud.pcd
  → Open in CloudCompare: maintains grid indexing via WIDTH/HEIGHT


In [5]:
def save_ply_with_grid(points, filename, colors=None, mask=None):
    H, W, _ = points.shape
    points_flat = points.reshape(-1, 3)
    rows, cols = np.mgrid[:H, :W].reshape(2, -1)

    if mask is not None:
        mask_flat = mask.reshape(-1)
        valid = np.where(mask_flat)[0]
        points_flat = points_flat[valid]
        rows, cols = rows[valid], cols[valid]
        if colors is not None:
            colors = colors.reshape(-1, 3)[valid]
    else:
        if colors is not None:
            colors = colors.reshape(-1, 3)

    n = len(points_flat)
    with open(filename, 'w') as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("property int row\nproperty int col\n")  # ← grid indices
        if colors is not None:
            f.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        f.write("end_header\n")
        if colors is not None:
            for (x,y,z), r, c, (R,G,B) in zip(points_flat, rows, cols, colors):
                f.write(f"{x} {y} {z} {r} {c} {int(R)} {int(G)} {int(B)}\n")
        else:
            for (x,y,z), r, c in zip(points_flat, rows, cols):
                f.write(f"{x} {y} {z} {r} {c}\n")
    print(f"✓ Saved {n} points with grid indices to {filename}")

# Usage example with your variables:
points = output["points"].cpu().numpy()      # (736, 1132, 3)
mask = output["mask"].cpu().numpy()          # (736, 1132) - boolean or 0/1
input_image_uint8 = (input_image * 255).astype(np.uint8) if input_image.max() <= 1.0 else input_image.astype(np.uint8)

# Save with colors and mask filtering
save_ply_with_grid(
    points=points,
    filename="pointcloud.ply",
    colors=input_image_uint8,
    mask=mask
)

✓ Saved 5379584 points with grid indices to pointcloud.ply


In [ ]:
colors_packed # array([ 98,  95, 101, ...,   0,   0,   0], dtype=uint8)

array([ 98,  95, 101, ...,   0,   0,   0], dtype=uint8)

In [6]:
import k3d
import numpy as np

# Prepare data
points = output["points"].cpu().numpy()          # (H, W, 3)
mask = output["mask"].cpu().numpy()              # (H, W)
rgb = input_image.copy()

# Normalize to uint8 [0,255] if needed
if rgb.dtype != np.uint8:
    rgb = (rgb * 255).astype(np.uint8)

# Flatten while preserving structure
H, W = points.shape[:2]
points_flat = points.reshape(-1, 3)
colors_flat = rgb.reshape(-1, 3)
mask_flat = mask.reshape(-1).astype(bool)

# Apply mask
points_viz = points_flat[mask_flat]
colors_viz = colors_flat[mask_flat]

# Pack RGB into single uint32 value per point (K3D format)
# Format: 0x00RRGGBB  (R in bits 16-23, G in 8-15, B in 0-7)
colors_packed = (
    (colors_viz[:, 0].astype(np.uint32) << 16) |  # Red
    (colors_viz[:, 1].astype(np.uint32) << 8)  |  # Green
    (colors_viz[:, 2].astype(np.uint32))          # Blue
)

print(f"Points: {len(points_viz)}, Colors dtype: {colors_packed.dtype}, sample: {colors_packed[:5]}")
# Should show large numbers like [6543590, 6481765, ...], NOT small [98, 95, ...]

# Create plot
plot = k3d.plot(grid_visible=False, camera_auto_fit=True)
pt = k3d.points(
    positions=points_viz.astype(np.float32),
    colors=colors_packed,
    point_size=0.001,
    shader='3d'  # 'flat' for faster rendering, '3d' for spherical points
)
plot += pt
plot.display()

Points: 5379584, Colors dtype: uint32, sample: [2629410 2629410 2432031 2234652 2234141]


Output()

In [ ]:
# save points to .ply file for visualization in CloudCompare or MeshLab
print(input_image.shape) # (736, 1132, 3)
print(output["points"].cpu().numpy().shape) # (736, 1132, 3)
points = output["points"].cpu().numpy()
mask = output["mask"].cpu().numpy()
# if all pixels are valid, save as dense point cloud; otherwise, save only valid points.
valid_points = points[mask > 0]
with open("output_points.ply", "w") as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(valid_points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("end_header\n")
    for p in valid_points:
        f.write(f"{p[0]} {p[1]} {p[2]}\n")

In [5]:
cv2.imshow("Depth Map", output["depth"].cpu().numpy() / output["depth"].max().item())
cv2.waitKey(0)
cv2.destroyAllWindows()